# Cross-node (federated) fairness audit — synthetic demonstration

Runs `fairscope`'s `FederatedFairnessAudit` on a **synthetic** set of three sites with different discrimination. **This audits per-node predictions only** — it does **not** train any model, perform secure aggregation or differential privacy, and makes **no privacy guarantee**. The numbers illustrate the audit, not a real system.

In [ ]:
import matplotlib

matplotlib.use('Agg')
import numpy as np

from fairscope.federated import FederatedFairnessAudit

rng = np.random.default_rng(20260627)

def node(n, sep):
    half = n // 2
    y = np.r_[np.ones(half, int), np.zeros(half, int)]
    raw = np.r_[rng.normal(sep, 1.0, half), rng.normal(0.0, 1.0, half)]
    return y, 1.0 / (1.0 + np.exp(-raw))

nodes = {
    'site_a': node(800, 1.4),  # strong
    'site_b': node(800, 1.0),
    'site_c': node(800, 0.4),  # weak -> cross-node gap
}

In [ ]:
report = FederatedFairnessAudit(nodes).run()
report.to_dataframe()

Per-node AUC (with DeLong CIs), ECE, Brier and F1. The summary flags the largest cross-node gap and any Bonferroni-significant pairwise difference.

In [ ]:
print(report.summary())

In [ ]:
report.disparity()

## Per-node recalibration

Temperature scaling / isotonic regression per node (from `core`), reporting pre/post ECE. This is an in-sample diagnostic; use a held-out split for a deployment estimate.

In [ ]:
report.recalibrate(method='temperature')

In [ ]:
report.plot_auc_forest()

In [ ]:
report.plot_calibration()

The cross-node AUC gap is a **diagnostic**, not a compliance verdict — and, again, this layer audits predictions; it provides no training or privacy guarantee.